# 08 — Neural-network spike early-warning model

This notebook builds a first neural-network workflow for predicting respiratory-virus spikes. It uses the repository's canonical predictive/predicted series, constructs leakage-aware spike labels, and fits scikit-learn MLP models for:

- probability of a spike within the next 1–4 weeks;
- expected spike severity above a train-period threshold;
- bootstrap uncertainty intervals for the severity score.

The modelling target is deliberately a spike event rather than ordinary case-count regression.


## Modelling definition

For each target series and prediction origin `t`, the future target is the maximum smoothed value over weeks `t+1, ..., t+h`. A spike is labelled when that future maximum exceeds a high historical threshold estimated from the training period only. Severity is the amount above that threshold, measured in robust training-scale units.

The feature matrix uses positive lags only, so predictors are restricted to information available before the prediction origin.


In [3]:
import sys
from pathlib import Path
from typing import Union


import pandas as pd


def find_repo_root_from_cwd(start: Union[int, None] = None) -> Path:
    """Find the repository root without requiring the wastewater package import."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "sources.csv").exists() and (candidate / "src" / "wastewater").exists():
            return candidate
    raise FileNotFoundError("Could not find repository root containing sources.csv and src/wastewater")


root = find_repo_root_from_cwd()
src_path = root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from wastewater.regression_matrix import build_available_series, summarise_available_series
from wastewater.spike_neural_network import (
    SpikeNNConfig,
    run_spike_neural_network_experiment,
    save_spike_outputs,
)

root


PosixPath('/home/malachy/wastewater-pathogen-data')

In [4]:
series = build_available_series(root)
summary = summarise_available_series(series)

print(f"Loaded {series['series_id'].nunique() if not series.empty else 0} canonical series")
summary.head(20)


/home/malachy/wastewater-pathogen-data/src/wastewater/io.py:60: DtypeWarning: Columns (19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep=sep, **kwargs)
/home/malachy/wastewater-pathogen-data/src/wastewater/regression_matrix.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce")
/home/malachy/wastewater-pathogen-data/src/wastewater/regression_matrix.py:156: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce")
/home/malachy/wastewater-pathogen-data/src/wastewater/regression_matrix.py:156: UserWarning: Could not infer format, so each element will be parsed indivi

Loaded 48 canonical series


,role,dataset_family,series_id,series_name,source_file,start_date,end_date,n_observations
0,predicted,ukhsa_gp_admissions,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,ukhsa-chart-Influenza-Hospital-Admissions-By-Week,ukhsa-chart-Influenza-Hospital-Admissions-By-W...,2017-03-06,2026-04-20,477
1,predicted,ukhsa_gp_admissions,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,ukhsa-chart-Influenza-Like-Syndromic-Emergency...,ukhsa-chart-Influenza-Like-Syndromic-Emergency...,2025-06-22,2026-06-21,1086
2,predicted,ukhsa_gp_admissions,ukhsa_gp_admissions::ukhsa-chart-Lower-Respira...,ukhsa-chart-Lower-Respiratory-Tract-Infection-...,ukhsa-chart-Lower-Respiratory-Tract-Infection-...,2025-06-18,2026-06-22,1093
3,predicted,ukhsa_gp_admissions,ukhsa_gp_admissions::ukhsa-chart-Upper-Respira...,ukhsa-chart-Upper-Respiratory-Tract-Infection-...,ukhsa-chart-Upper-Respiratory-Tract-Infection-...,2025-06-18,2026-06-22,728
4,predictive,google_trends_1y,google_trends_1y::20250630 1431 20260630 1431,20250630 1431 20260630 1431,Google_trends_v2/1y_data/time_series_GB_202506...,2025-06-29,2026-06-28,53
5,predictive,google_trends_1y,google_trends_1y::20250630 1431 20260630 1431 (1),20250630 1431 20260630 1431 (1),Google_trends_v2/1y_data/time_series_GB_202506...,2025-06-29,2026-06-28,53
6,predictive,google_trends_1y,google_trends_1y::20250630 1432 20260630 1432,20250630 1432 20260630 1432,Google_trends_v2/1y_data/time_series_GB_202506...,2025-06-29,2026-06-28,53
7,predictive,google_trends_1y,google_trends_1y::20250630 1432 20260630 1432 (1),20250630 1432 20260630 1432 (1),Google_trends_v2/1y_data/time_series_GB_202506...,2025-06-29,2026-06-28,53
8,predictive,google_trends_1y,google_trends_1y::20250630 1432 20260630 1432 (2),20250630 1432 20260630 1432 (2),Google_trends_v2/1y_data/time_series_GB_202506...,2025-06-29,2026-06-28,53
9,predictive,google_trends_1y,google_trends_1y::20250630 1433 20260630 1433,20250630 1433 20260630 1433,Google_trends_v2/1y_data/time_series_GB_202506...,2025-06-29,2026-06-28,53


## Choose targets

By default this notebook prioritises all available predicted series. If the run is slow, set `MAX_TARGETS` to a small number while developing.


In [5]:
target_ids = sorted(series.loc[series["role"] == "predicted", "series_id"].dropna().unique())

# Keep this as None for a full run, or use a small integer for a quick smoke test.
MAX_TARGETS = None
selected_target_ids = target_ids[:MAX_TARGETS] if MAX_TARGETS is not None else target_ids

print(f"Selected {len(selected_target_ids)} target series")
selected_target_ids[:10]


Selected 4 target series


['ukhsa_gp_admissions::ukhsa-chart-Influenza-Hospital-Admissions-By-Week',
 'ukhsa_gp_admissions::ukhsa-chart-Influenza-Like-Syndromic-Emergency-Department-Admissions-Daily',
 'ukhsa_gp_admissions::ukhsa-chart-Lower-Respiratory-Tract-Infection-GP-Hours-Daily',
 'ukhsa_gp_admissions::ukhsa-chart-Upper-Respiratory-Tract-Infection-GP-Hours-Daily']

## Fit spike neural networks

This uses an MLP classifier for spike probability and a bootstrap ensemble of MLP regressors for severity intervals. The bootstrap interval is a practical uncertainty estimate rather than a fully calibrated epidemiological prediction interval.


In [6]:
config = SpikeNNConfig(
    horizons=(1, 2, 3, 4),
    lags=(1, 2, 3, 4, 6, 8, 12),
    threshold_quantile=0.85,
    smoothing_window=3,
    interval_level=0.80,
    hidden_layer_sizes=(64, 32),
    n_bootstrap_models=25,
    min_train_rows=24,
    min_test_rows=4,
)

results, predictions = run_spike_neural_network_experiment(
    series,
    target_ids=selected_target_ids,
    config=config,
    train_fraction=0.8,
    random_state=42,
)

result_path, prediction_path = save_spike_outputs(results, predictions, root)
print(result_path)
print(prediction_path)
results


/home/malachy/wastewater-pathogen-data/src/wastewater/spike_neural_network.py:254: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = source.shift(lag)
/home/malachy/wastewater-pathogen-data/src/wastewater/spike_neural_network.py:254: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel[col] = source.shift(lag)
/home/malachy/wastewater-pathogen-data/src/wastewater/spike_neural_network.py:254: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

/home/malachy/wastewater-pathogen-data/data/processed/respiratory_spike_neural_network_results.csv
/home/malachy/wastewater-pathogen-data/data/processed/respiratory_spike_neural_network_predictions.csv


,target_id,horizon,status,classifier,severity_model,n_train,n_test,n_features,train_start,train_end,...,severity_interval_coverage,severity_interval_width,spike_threshold,severity_scale,brier,precision,recall,f1,average_precision,roc_auc
0,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,1,ok,mlp_classifier,bootstrap_mlp_regressor_25,382,96,56,1969-12-29,2024-06-17,...,0.770833,1.852891,2.083333,0.833333,0.135501,0.875000,0.600000,0.711864,0.838412,0.867916
1,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,2,ok,mlp_classifier,bootstrap_mlp_regressor_25,381,96,56,1969-12-29,2024-06-10,...,0.812500,2.040190,2.115833,0.834167,0.200437,1.000000,0.351351,0.520000,0.803499,0.812185
2,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,3,ok,mlp_classifier,bootstrap_mlp_regressor_25,380,96,56,1969-12-29,2024-06-03,...,0.812500,2.469129,2.148333,0.835000,0.237268,0.652174,0.384615,0.483871,0.710012,0.762483
3,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,4,ok,mlp_classifier,bootstrap_mlp_regressor_25,379,96,56,1969-12-29,2024-05-27,...,0.750000,2.879992,2.180833,0.835833,0.202356,0.800000,0.390244,0.524590,0.815334,0.846120
4,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,1,ok,mlp_classifier,bootstrap_mlp_regressor_25,42,11,252,2025-06-09,2026-03-23,...,1.000000,1.835789,180.939683,98.466349,0.020590,0.000000,0.000000,0.000000,NaN,NaN
5,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,2,ok,mlp_classifier,bootstrap_mlp_regressor_25,41,11,252,2025-06-02,2026-03-09,...,1.000000,0.964964,192.052587,106.177698,0.118961,0.000000,0.000000,0.000000,NaN,NaN
6,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,3,ok,mlp_classifier,bootstrap_mlp_regressor_25,40,11,252,2025-05-26,2026-02-23,...,1.000000,2.229701,203.165492,113.889048,0.100222,0.000000,0.000000,0.000000,NaN,NaN
7,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,4,ok,mlp_classifier,bootstrap_mlp_regressor_25,39,11,252,2025-05-19,2026-02-09,...,1.000000,1.768800,214.278397,118.672063,0.003054,0.000000,0.000000,0.000000,NaN,NaN
8,ukhsa_gp_admissions::ukhsa-chart-Lower-Respira...,1,ok,mlp_classifier,bootstrap_mlp_regressor_25,42,11,252,2025-06-09,2026-03-23,...,1.000000,1.816868,15.196349,7.297619,0.021943,0.000000,0.000000,0.000000,NaN,NaN
9,ukhsa_gp_admissions::ukhsa-chart-Lower-Respira...,2,ok,mlp_classifier,bootstrap_mlp_regressor_25,41,11,252,2025-06-02,2026-03-09,...,1.000000,0.966371,15.445540,7.351032,0.089074,0.000000,0.000000,0.000000,NaN,NaN


## Inspect model quality

For spike prediction, average precision, recall, Brier score, and interval coverage are often more informative than ordinary accuracy.


In [7]:
ok = results[results["status"] == "ok"].copy()
metric_cols = [
    "target_id",
    "horizon",
    "n_train",
    "n_test",
    "train_spike_rate",
    "test_spike_rate",
    "average_precision",
    "roc_auc",
    "brier",
    "precision",
    "recall",
    "f1",
    "severity_mae",
    "severity_interval_coverage",
    "severity_interval_width",
]
ok[metric_cols].sort_values(["average_precision", "recall"], ascending=False)


,target_id,horizon,n_train,n_test,train_spike_rate,test_spike_rate,average_precision,roc_auc,brier,precision,recall,f1,severity_mae,severity_interval_coverage,severity_interval_width
0,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,1,382,96,0.151832,0.364583,0.838412,0.867916,0.135501,0.875000,0.600000,0.711864,1.145249,0.770833,1.852891
3,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,4,379,96,0.189974,0.427083,0.815334,0.846120,0.202356,0.800000,0.390244,0.524590,1.690788,0.750000,2.879992
1,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,2,381,96,0.162730,0.385417,0.803499,0.812185,0.200437,1.000000,0.351351,0.520000,1.268414,0.812500,2.040190
2,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,3,380,96,0.176316,0.406250,0.710012,0.762483,0.237268,0.652174,0.384615,0.483871,1.459364,0.812500,2.469129
4,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,1,42,11,0.166667,0.000000,NaN,NaN,0.020590,0.000000,0.000000,0.000000,0.627508,1.000000,1.835789
5,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,2,41,11,0.170732,0.000000,NaN,NaN,0.118961,0.000000,0.000000,0.000000,0.288717,1.000000,0.964964
6,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,3,40,11,0.200000,0.000000,NaN,NaN,0.100222,0.000000,0.000000,0.000000,0.786165,1.000000,2.229701
7,ukhsa_gp_admissions::ukhsa-chart-Influenza-Lik...,4,39,11,0.230769,0.000000,NaN,NaN,0.003054,0.000000,0.000000,0.000000,0.567484,1.000000,1.768800
8,ukhsa_gp_admissions::ukhsa-chart-Lower-Respira...,1,42,11,0.166667,0.000000,NaN,NaN,0.021943,0.000000,0.000000,0.000000,0.624076,1.000000,1.816868
9,ukhsa_gp_admissions::ukhsa-chart-Lower-Respira...,2,41,11,0.170732,0.000000,NaN,NaN,0.089074,0.000000,0.000000,0.000000,0.288101,1.000000,0.966371


## Inspect predicted warnings

The table below surfaces out-of-sample weeks with the highest predicted spike probability. The severity interval is expressed in training-scale units above the spike threshold.


In [8]:
warning_cols = [
    "period",
    "target_id",
    "horizon",
    "future_window_end",
    "spike_probability",
    "predicted_spike",
    "predicted_severity_score",
    "severity_score_lower",
    "severity_score_upper",
    "predicted_severity_band",
    "severity_band_lower",
    "severity_band_upper",
    "spike",
    "severity_score",
]

predictions[warning_cols].sort_values("spike_probability", ascending=False).head(30)


,period,target_id,horizon,future_window_end,spike_probability,predicted_spike,predicted_severity_score,severity_score_lower,severity_score_upper,predicted_severity_band,severity_band_lower,severity_band_upper,spike,severity_score
32,2025-02-10,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,1,2025-02-17,0.997224,1,5.013104,3.067213,7.029774,severe,severe,severe,1,3.648000
31,2025-02-03,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,1,2025-02-10,0.993881,1,6.164845,3.189924,8.606289,severe,severe,severe,1,4.408000
227,2025-03-03,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,3,2025-03-24,0.993785,1,2.566056,0.179108,5.096706,severe,borderline,severe,1,1.475050
30,2025-01-27,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,1,2025-02-03,0.993363,1,5.056444,2.334079,7.151093,severe,severe,severe,1,5.452000
29,2025-01-20,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,1,2025-01-27,0.990882,1,5.683164,2.373306,8.229827,severe,severe,severe,1,7.072000
123,2025-01-06,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,2,2025-01-20,0.990747,1,8.125300,5.681031,10.894166,severe,severe,severe,1,11.601399
315,2025-01-06,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,4,2025-02-03,0.988265,1,10.152268,5.755275,14.505822,severe,severe,severe,1,11.500499
33,2025-02-17,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,1,2025-02-24,0.987352,1,6.214331,3.737752,9.908046,severe,severe,severe,1,2.992000
228,2025-03-10,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,3,2025-03-31,0.985910,1,1.795337,0.000000,3.967685,moderate,none,severe,1,1.163673
124,2025-01-13,ukhsa_gp_admissions::ukhsa-chart-Influenza-Hos...,2,2025-01-27,0.983858,1,5.930192,2.653640,9.353341,severe,severe,severe,1,8.636364


## Next modelling improvements

- Calibrate spike probabilities with rolling-origin backtests rather than a single chronological split.
- Add explicit regional and virus identifiers when the panel is expanded to multiple geographies/pathogens.
- Replace the MLP with a temporal convolutional network once PyTorch or another deep-learning dependency is added.
- Evaluate operational utility as true spikes detected at least two weeks early at a fixed false-alert budget.
